# Modelling: sentiment classifier

TF-IDF features + a linear classifier. I compare logistic regression, Naive Bayes and linear SVM with cross-validation, then keep one model.

Cleaning lives inside the pipeline (`TfidfVectorizer(preprocessor=clean_text)`), so the same transformation runs at train and serve time. This notebook mirrors `sentiment_classifier.train`; run that script to regenerate the saved artifact.

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
from sentiment_classifier.preprocessing import clean_text

df = pd.read_csv("../data/IMDB_reviews.csv").drop_duplicates(subset="review")
len(df)

Stratified 80/20 split on the deduplicated data.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    df["review"], df["sentiment"], test_size=0.2,
    stratify=df["sentiment"], random_state=42)

def make_pipeline(estimator):
    vectorizer = TfidfVectorizer(preprocessor=clean_text, stop_words="english",
                                 ngram_range=(1, 2), min_df=5, max_df=0.95,
                                 max_features=20000)
    return Pipeline([("tfidf", vectorizer), ("clf", estimator)])

## Model comparison (5-fold CV on the training set)

In [ ]:
candidates = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42),
    "Naive Bayes": MultinomialNB(),
    "Linear SVC": LinearSVC(random_state=42),
}
for name, est in candidates.items():
    cv = cross_val_score(make_pipeline(est), X_train, y_train, cv=5, n_jobs=-1)
    print(f"{name:<22} {cv.mean():.4f} +/- {cv.std():.4f}")

Logistic regression and linear SVM are tied (~0.90) and both beat Naive Bayes. I keep **logistic regression**: same accuracy, and it exposes `predict_proba`, which the demo uses to show a confidence score.

In [ ]:
model = make_pipeline(LogisticRegression(max_iter=1000, random_state=42))
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
print(classification_report(y_test, y_pred))

In [ ]:
cm = confusion_matrix(y_test, y_pred, labels=model.classes_)
ConfusionMatrixDisplay(cm, display_labels=model.classes_).plot(cmap="Blues")

## Error inspection

The most confident mistakes, useful to understand where a bag-of-words model breaks (sarcasm, negation, mixed reviews).

In [ ]:
import numpy as np
proba = model.predict_proba(X_test).max(axis=1)
wrong = (y_pred != y_test.values)
idx = np.argsort(np.where(wrong, proba, -1))[-3:]
for i in idx:
    print(f"true={y_test.values[i]} pred={y_pred[i]} p={proba[i]:.2f}")
    print(X_test.values[i][:300], "...\n")

The saved artifact (`models/sentiment_pipeline.joblib`) is produced by `python -m sentiment_classifier.train`, which runs exactly this pipeline.